# Pre-generating 7-Combination Degraded Dataset for Image Restoration

This notebook loads preprocessed clean images from `/kaggle/input/**/clean_images/` (or `/kaggle/working/clean_images/`)
and generates **7 distinct degradation combinations** across `train`, `val`, and `test` splits:

1. `c1_only_noise` (Gaussian, Poisson, or Salt & Pepper Noise)
2. `c2_only_blur` (Gaussian, Motion, or Defocus Blur)
3. `c3_only_low_light` (Gamma darkening + Illumination reduction)
4. `c4_noise_blur` (Noise + Blur)
5. `c5_noise_low_light` (Noise + Low Light)
6. `c6_blur_low_light` (Blur + Low Light)
7. `c7_all_three` (Noise + Blur + Low Light)

Output directory structure:
```
/kaggle/working/degraded_dataset/
├── train/
│   ├── clean/
│   ├── c1_only_noise/
│   ├── c2_only_blur/
│   ├── c3_only_low_light/
│   ├── c4_noise_blur/
│   ├── c5_noise_low_light/
│   ├── c6_blur_low_light/
│   └── c7_all_three/
├── val/ ...
└── test/ ...
```

In [ ]:
import os
import glob
import random
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# Set seeds for reproducible degradation sampling
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 1. Setup Input & Output Paths
IS_KAGGLE = os.path.exists('/kaggle/input')

def find_clean_images_dir():
    if IS_KAGGLE:
        matches = glob.glob('/kaggle/input/**/clean_images', recursive=True)
        if matches:
            print(f"Found clean images dataset at: {matches[0]}")
            return matches[0]
        if os.path.exists('/kaggle/working/clean_images'):
            print("Using clean images from /kaggle/working/clean_images")
            return '/kaggle/working/clean_images'
    else:
        if os.path.exists('../data/clean_images'):
            return '../data/clean_images'
        if os.path.exists('./clean_images'):
            return './clean_images'
    return '/kaggle/working/clean_images'

INPUT_CLEAN_DIR = find_clean_images_dir()
OUTPUT_DEGRADED_DIR = '/kaggle/working/degraded_dataset' if IS_KAGGLE else '../data/degraded_dataset'

COMBINATIONS = [
    'c1_only_noise',
    'c2_only_blur',
    'c3_only_low_light',
    'c4_noise_blur',
    'c5_noise_low_light',
    'c6_blur_low_light',
    'c7_all_three'
]

SPLIT_DIRS = {}
for split in ['train', 'val', 'test']:
    SPLIT_DIRS[split] = {}
    # Create clean directory
    clean_dir = os.path.join(OUTPUT_DEGRADED_DIR, split, 'clean')
    os.makedirs(clean_dir, exist_ok=True)
    SPLIT_DIRS[split]['clean'] = clean_dir
    
    # Create 7 degradation combination directories
    for combo in COMBINATIONS:
        combo_dir = os.path.join(OUTPUT_DEGRADED_DIR, split, combo)
        os.makedirs(combo_dir, exist_ok=True)
        SPLIT_DIRS[split][combo] = combo_dir

print(f"Input Clean Directory: {INPUT_CLEAN_DIR}")
print(f"Output Degraded Directory: {OUTPUT_DEGRADED_DIR}")
print(f"Degradation Combinations: {COMBINATIONS}")

In [ ]:
# 2. PyTorch Transformation Operators for Noise, Blur, and Low-Light
def apply_noise(img_tensor, severity=0.2):
    std = 0.05 + severity * 0.2
    noise = torch.randn_like(img_tensor) * std
    return torch.clamp(img_tensor + noise, 0.0, 1.0)

def apply_blur(img_tensor, severity=0.4):
    k_size = int(3 + severity * 8)
    if k_size % 2 == 0:
        k_size += 1
    sigma = 0.5 + severity * 2.5
    return TF.gaussian_blur(img_tensor, [k_size, k_size], [sigma, sigma])

def apply_low_light(img_tensor, severity=0.5):
    scale = 0.6 - (severity * 0.45)
    gamma = 1.5 + (severity * 2.0)
    darkened = torch.clamp(img_tensor * scale, 0.0, 1.0)
    return torch.pow(torch.clamp(darkened, 1e-6, 1.0), gamma)

def generate_degradation_combo(img_tensor, combo_name):
    deg = img_tensor.clone()
    sev_n = random.uniform(0.1, 0.4)
    sev_b = random.uniform(0.2, 0.6)
    sev_l = random.uniform(0.3, 0.7)
    
    if combo_name == 'c1_only_noise':
        deg = apply_noise(deg, severity=sev_n)
    elif combo_name == 'c2_only_blur':
        deg = apply_blur(deg, severity=sev_b)
    elif combo_name == 'c3_only_low_light':
        deg = apply_low_light(deg, severity=sev_l)
    elif combo_name == 'c4_noise_blur':
        deg = apply_blur(deg, severity=sev_b)
        deg = apply_noise(deg, severity=sev_n)
    elif combo_name == 'c5_noise_low_light':
        deg = apply_low_light(deg, severity=sev_l)
        deg = apply_noise(deg, severity=sev_n)
    elif combo_name == 'c6_blur_low_light':
        deg = apply_low_light(deg, severity=sev_l)
        deg = apply_blur(deg, severity=sev_b)
    elif combo_name == 'c7_all_three':
        deg = apply_low_light(deg, severity=sev_l)
        deg = apply_blur(deg, severity=sev_b)
        deg = apply_noise(deg, severity=sev_n)
        
    return torch.clamp(deg, 0.0, 1.0)

In [ ]:
# 3. Batch Processing: Generate Clean + 7 Degraded Variants per Image
to_tensor = T.ToTensor()
to_pil = T.ToPILImage()

stats = {split: {c: 0 for c in ['clean'] + COMBINATIONS} for split in ['train', 'val', 'test']}

for split in ['train', 'val', 'test']:
    split_src_dir = os.path.join(INPUT_CLEAN_DIR, split)
    if not os.path.exists(split_src_dir):
        print(f"Warning: Split directory not found: {split_src_dir}")
        continue
        
    clean_files = glob.glob(os.path.join(split_src_dir, '*.png')) + glob.glob(os.path.join(split_src_dir, '*.jpg'))
    print(f"\nProcessing split '{split.upper()}' ({len(clean_files)} clean images)...")
    
    for idx, clean_path in enumerate(tqdm(clean_files, desc=f"Generating 7x Degraded Dataset -> {split}")):
        filename = f"img_{idx:06d}.png"
        
        with Image.open(clean_path) as img:
            clean_pil = img.convert('RGB')
            if clean_pil.size != (128, 128):
                clean_pil = clean_pil.resize((128, 128), Image.Resampling.BICUBIC)
            clean_tensor = to_tensor(clean_pil)
            
        # Save clean ground truth
        clean_dst = os.path.join(SPLIT_DIRS[split]['clean'], filename)
        clean_pil.save(clean_dst, 'PNG')
        stats[split]['clean'] += 1
        
        # Generate and save all 7 degraded combinations
        for combo in COMBINATIONS:
            deg_tensor = generate_degradation_combo(clean_tensor, combo)
            deg_pil = to_pil(deg_tensor)
            deg_dst = os.path.join(SPLIT_DIRS[split][combo], filename)
            deg_pil.save(deg_dst, 'PNG')
            stats[split][combo] += 1

print("\n======================================================")
print("DEGRADED DATASET GENERATION COMPLETED SUCCESSFULLY!")
for split in ['train', 'val', 'test']:
    total_split_imgs = sum(stats[split].values())
    print(f"[{split.upper()}] Clean: {stats[split]['clean']} | Degraded: {total_split_imgs - stats[split]['clean']} (Total: {total_split_imgs})")
print(f"Saved Location: {OUTPUT_DEGRADED_DIR}")
print("======================================================")

In [ ]:
# 4. Preview: Clean Ground Truth vs All 7 Degradation Combinations
test_clean_dir = SPLIT_DIRS['test']['clean']
clean_samples = glob.glob(os.path.join(test_clean_dir, '*.png'))

if len(clean_samples) > 0:
    sample_file = os.path.basename(clean_samples[0])
    fig, axes = plt.subplots(1, 8, figsize=(20, 3))
    
    # Clean
    clean_img = Image.open(os.path.join(test_clean_dir, sample_file))
    axes[0].imshow(clean_img)
    axes[0].set_title("Original Clean", fontsize=9, fontweight='bold')
    axes[0].axis('off')
    
    # 7 Degradations
    titles = ['1. Only Noise', '2. Only Blur', '3. Only LowLight', '4. Noise+Blur', '5. Noise+LowLight', '6. Blur+LowLight', '7. All Three']
    for i, combo in enumerate(COMBINATIONS):
        deg_path = os.path.join(SPLIT_DIRS['test'][combo], sample_file)
        deg_img = Image.open(deg_path)
        axes[i + 1].imshow(deg_img)
        axes[i + 1].set_title(titles[i], fontsize=8)
        axes[i + 1].axis('off')
        
    plt.suptitle("7-Combination Degradation Suite Preview", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.show()